In [15]:
import pandas as pd
import numpy as np
import warnings, re
warnings.filterwarnings('ignore')

In [16]:
bouts = pd.read_csv('../data/sumodb_all_bouts_final.csv')
players = pd.read_csv('../data\sumodb_names_batch_20250923_105158\sumodb_wrestlers_final.csv')

In [17]:
bouts

,basho,day,wrestler1,wrestler1_rank,wrestler1_result,wrestler2,wrestler2_rank,wrestler2_result,kimarite,scraped_at,source_offset
0,2000.01,1,Aminishiki,J13w,1-0 (8-7),Wakakosho,J13e,0-1 (8-7),yorikiri,2025-09-22T18:52:24.380838,0
1,2000.01,1,Kitazakura,J12w,1-0 (7-8),Takamisakari,J12e,0-1 (7-8),yorikiri,2025-09-22T18:52:24.380838,0
2,2000.01,1,Tochinohana,J11e,1-0 (9-6),Kobo,J11w,0-1 (7-8),yoritaoshi,2025-09-22T18:52:24.380838,0
3,2000.01,1,Tamanonada,J10w,1-0 (5-10),Ganyu,J10e,0-1 (7-8),yorikiri,2025-09-22T18:52:24.380838,0
4,2000.01,1,Tomonohana,J9e,1-0 (9-6),Sentoryu,J9w,0-1 (7-8),shitatedashinage,2025-09-22T18:52:24.380838,0
...,...,...,...,...,...,...,...,...,...,...,...
76063,2025.09,9,Hakuoho,M2e,6-3,Takayasu,K1e,2-7,yorikiri,2025-09-22T19:03:59.717224,76000
76064,2025.09,9,Kirishima,S1w,5-4,Wakatakakage,S1e,5-4,sukuinage,2025-09-22T19:03:59.717224,76000
76065,2025.09,9,Kotozakura,O1e,6-3,Atamifuji,M3e,2-7,katasukashi,2025-09-22T19:03:59.717224,76000
76066,2025.09,9,Hoshoryu,Y1wYO,9-0,Kotoshoho,M5e,2-7,tsukiotoshi,2025-09-22T19:03:59.717224,76000


In [18]:
players

,rikishi,heya,shusshin,birth_date,hatsu,intai,height,weight,highest_rank,career_high,date_info,rank_info,age_info,record_info,scraped_at,source_offset
0,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1966.11,Sd91w,18.1,2-5,2025-09-23T10:52:07.501629,100000
1,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1967.01,Jd15e,19.0,4-3,2025-09-23T10:52:07.502630,100000
2,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1967.03,Sd91e,19.2,3-4,2025-09-23T10:52:07.502630,100000
3,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1967.05,Jd59e,19.4,5-2,2025-09-23T10:52:07.502630,100000
4,Takunishiki,Mihogaseki,Saga,19.12.1947,1964.05,1972.11,177.0,97.0,Sd91,Ms41,1967.07,Jd5w,19.6,1-6,2025-09-23T10:52:07.502630,100000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247924,Sawazaki,Onoe,Kagoshima,06.03.2010,2025.03,NaN,166.0,107.0,Jd87,Jd87,2025.07,Jd87e,15.4,0-7,2025-09-23T11:37:51.009213,348000
247925,Sawazaki,Onoe,Kagoshima,06.03.2010,2025.03,NaN,166.0,107.0,Jd87,Jd87,2025.09,Jk13e,15.6,3-2,2025-09-23T11:37:51.009213,348000
247926,Tosoumi,Tamanoi,Iwate,23.03.2010,2025.05,NaN,178.0,113.0,Mz,Jk17,2025.05,Mz,15.1,2-3,2025-09-23T11:37:51.009213,348000
247927,Tosoumi,Tamanoi,Iwate,23.03.2010,2025.05,NaN,178.0,113.0,Jk24,Jk17,2025.07,Jk24e,15.3,3-4,2025-09-23T11:37:51.010214,348000


In [19]:
# Create a stacked bouts that shows win vs loss. Right now, wrestler1 is always the winner
bouts['bout_id'] = np.arange(len(bouts))

bouts_win = bouts[['basho', 'day', 'wrestler1', 'wrestler1_rank', 'wrestler1_result',
       'wrestler2', 'wrestler2_rank', 'wrestler2_result', 'kimarite', 'bout_id']]
bouts_win['win'] = 1
bouts_loss = bouts[['basho', 'day', 'wrestler1', 'wrestler1_rank', 'wrestler1_result',
       'wrestler2', 'wrestler2_rank', 'wrestler2_result', 'kimarite', 'bout_id']]
bouts_loss['win'] = 0

bouts_win.columns = ['basho', 'day', 'name', 'rank', 'tournament_record', 'opponent_name', 'opponent_rank', 'opponent_tournament_record', 'technique', 'bout_id', 'win']
bouts_loss.columns = ['basho', 'day', 'opponent_name', 'opponent_rank', 'opponent_tournament_record', 'name', 'rank', 'tournament_record', 'technique', 'bout_id', 'win']

bouts_all = pd.concat([bouts_win, bouts_loss], ignore_index=True)
bouts_all

,basho,day,name,rank,tournament_record,opponent_name,opponent_rank,opponent_tournament_record,technique,bout_id,win
0,2000.01,1,Aminishiki,J13w,1-0 (8-7),Wakakosho,J13e,0-1 (8-7),yorikiri,0,1
1,2000.01,1,Kitazakura,J12w,1-0 (7-8),Takamisakari,J12e,0-1 (7-8),yorikiri,1,1
2,2000.01,1,Tochinohana,J11e,1-0 (9-6),Kobo,J11w,0-1 (7-8),yoritaoshi,2,1
3,2000.01,1,Tamanonada,J10w,1-0 (5-10),Ganyu,J10e,0-1 (7-8),yorikiri,3,1
4,2000.01,1,Tomonohana,J9e,1-0 (9-6),Sentoryu,J9w,0-1 (7-8),shitatedashinage,4,1
...,...,...,...,...,...,...,...,...,...,...,...
152131,2025.09,9,Takayasu,K1e,2-7,Hakuoho,M2e,6-3,yorikiri,76063,0
152132,2025.09,9,Wakatakakage,S1e,5-4,Kirishima,S1w,5-4,sukuinage,76064,0
152133,2025.09,9,Atamifuji,M3e,2-7,Kotozakura,O1e,6-3,katasukashi,76065,0
152134,2025.09,9,Kotoshoho,M5e,2-7,Hoshoryu,Y1wYO,9-0,tsukiotoshi,76066,0


In [20]:
# Removing low-ranked players
players = players[~players['rank_info'].isin(['Mz','Bg'])] # Mae-zumo and Banzuke-gai

players = players[~players['rank_info'].str.startswith('Jd', na=False)] # Jonidan
players = players[~players['rank_info'].str.startswith('Jk', na=False)] # Jonokuchi
players = players[~players['rank_info'].str.startswith('Sd', na=False)] # Sandanme

players['division'] = np.where(players['rank_info'].str.startswith('Ms', na=False), 'Makushita',
                      np.where(players['rank_info'].str.startswith('J', na=False), 'Juryo',
                      np.where(players['rank_info'].str.startswith('M', na=False), 'Maegashira',
                      np.where(players['rank_info'].str.startswith('Y', na=False), 'Maegashira',
                      np.where(players['rank_info'].str.startswith('K', na=False), 'Maegashira',
                      np.where(players['rank_info'].str.startswith('S', na=False), 'Maegashira',
                      np.where(players['rank_info'].str.startswith('O', na=False), 'Maegashira', ''
                      )))))))

In [21]:
# Create experience field
players['hatsu'] = players['hatsu'].astype(str).str.replace('.','/')
players['date2'] = players['date_info'].astype(str).str.replace('.','/')

players['debut_date'] = pd.to_datetime(players['hatsu'], errors='coerce', format='%Y/%m')
players['date2']       = pd.to_datetime(players['date2'] , errors='coerce', format='%Y/%m')
players['experience'] = (players['date2'] - players['debut_date']).dt.days / 365
players.drop(columns=['debut_date','date2'], inplace=True)

players['birth_date'] = pd.to_datetime(players['birth_date'], errors='coerce', format='%d.%m.%Y')

players.rename(columns={
    'date_info'   : 'date', 
    'rank_info'   : 'rank',
    'age_info'    : 'age', 
    'record_info' : 'record'
}, inplace=True)

In [22]:
# Creating a cross-division rank
def extract_letters_numbers(s):
    letters = re.search(r'[A-Za-z]+', str(s))
    numbers = re.search(r'\d+', str(s))
    return (letters.group(0) if letters else None, numbers.group(0) if numbers else None)

players[['rank_letters', 'rank_numbers']] = players['rank'].apply(lambda x: pd.Series(extract_letters_numbers(x)))

# Point of this modifier is to adjust the within-division ranks to reflect the number of ranks in the higher-ranking divisions
players['rank_letter_modifier'] = np.where(
    players['rank_letters'] == 'Y', 0, np.where(
    players['rank_letters'] == 'O', 2, np.where(
    players['rank_letters'] == 'S', 5, np.where(
    players['rank_letters'] == 'K', 7, np.where(
    players['rank_letters'] == 'M', 9, np.where(
    players['rank_letters'] == 'J', 27, np.where(
    players['rank_letters'] == 'Ms', 41, 999)))))))

# convert to numeric
players['rank_numbers'] = pd.to_numeric(players['rank_numbers'], errors='coerce')
players['rank_num'] = players['rank_numbers'] + players['rank_letter_modifier']
players.drop(columns=['rank_letters', 'rank_numbers', 'rank_letter_modifier'], inplace=True)

In [23]:
# Merge the player information on bouts_all
players1 = players.copy()
players2 = players.copy()
players2.columns = [col if col in ['rikishi', 'date','rank'] else 'opponent_'+col for col in players2.columns]

df_all = pd.merge(bouts_all, players1, left_on=['basho','name','rank'], right_on=['date','rikishi','rank'], how='left', validate='m:1', suffixes=(None,'_y'))
df_all.drop(columns=['date','rikishi'], inplace=True)
df_all = pd.merge(df_all, players2, left_on=['basho','opponent_name','opponent_rank'], right_on=['date','rikishi','rank'], how='left', validate='m:1', suffixes=(None,'_y'))
df_all.drop(columns=['date','rikishi','rank_y'], inplace=True)

In [24]:
df_all = df_all[[
    'basho', 'day', 'bout_id', 'technique', 'win',
    'name', 'heya', 'shusshin', 'birth_date', 'age', 'division', 
    'height', 'weight', 'experience', 'rank', 'rank_num', 
    'opponent_name', 'opponent_heya', 'opponent_shusshin', 'opponent_age', 'opponent_division', 
    'opponent_height', 'opponent_weight', 'opponent_experience', 'opponent_rank','opponent_rank_num'
]]
df_all

,basho,day,bout_id,technique,win,name,heya,shusshin,birth_date,age,...,opponent_name,opponent_heya,opponent_shusshin,opponent_age,opponent_division,opponent_height,opponent_weight,opponent_experience,opponent_rank,opponent_rank_num
0,2000.01,1,0,yorikiri,1,Aminishiki,Ajigawa,Aomori,1978-10-03,21.30,...,Wakakosho,Matsugane,Hyogo,24.1,Juryo,184.5,170.0,9.676712,J13e,40.0
1,2000.01,1,1,yorikiri,1,Kitazakura,Kitanoumi,Hiroshima,1971-12-15,28.00,...,Takamisakari,Azumazeki,Aomori,23.7,Juryo,187.0,138.0,0.838356,J12e,39.0
2,2000.01,1,2,yoritaoshi,1,Tochinohana,Kasugano,Iwate,1973-02-28,26.10,...,Kobo,Miyagino,Kagoshima,26.4,Juryo,182.5,121.5,10.843836,J11w,38.0
3,2000.01,1,3,yorikiri,1,Tamanonada,Kataonami,Fukushima,1977-09-15,22.30,...,Ganyu,Kitanoumi,Hyogo,29.5,Juryo,185.0,172.5,13.846575,J10e,37.0
4,2000.01,1,4,shitatedashinage,1,Tomonohana,Tatsunami,Kumamoto,1964-06-23,35.60,...,Sentoryu,Tomozuna,U.S.A.,30.5,Juryo,176.0,137.0,11.509589,J9w,36.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152131,2025.09,9,76063,yorikiri,0,Takayasu,Tagonoura,Ibaraki,1990-02-28,35.60,...,Hakuoho,Isegahama,Tottori,22.0,Maegashira,181.0,159.0,2.668493,M2e,11.0
152132,2025.09,9,76064,sukuinage,0,Wakatakakage,Arashio,Fukushima,1994-12-06,30.90,...,Kirishima,Otowayama,Mongolia,29.4,Maegashira,186.0,147.0,10.512329,S1w,6.0
152133,2025.09,9,76065,katasukashi,0,Atamifuji,Isegahama,Shizuoka,2002-09-03,23.00,...,Kotozakura,Sadogatake,Chiba,27.9,Maegashira,189.0,179.0,9.841096,O1e,3.0
152134,2025.09,9,76066,tsukiotoshi,0,Kotoshoho,Sadogatake,Chiba,1999-08-26,26.00,...,Hoshoryu,Tatsunami,Mongolia,26.3,Maegashira,188.0,150.0,7.838356,Y1wYO,1.0


In [25]:
# Create columns comparing player to opponent
df_all['age_diff'] = df_all['age'] - df_all['opponent_age']
df_all['height_diff'] = df_all['height'] - df_all['opponent_height']
df_all['weight_diff'] = df_all['weight'] - df_all['opponent_weight']
df_all['experience_diff'] = df_all['experience'] - df_all['opponent_experience']
df_all['rank_diff'] = df_all['rank_num'] - df_all['opponent_rank_num']
df_all['flag_higher_division'] = np.where(
    (df_all['division'] == 'Maegashira') & (df_all['opponent_division'] == 'Juryo')    , 1, np.where(
    (df_all['division'] == 'Juryo')      & (df_all['opponent_division'] == 'Makushita'), 1, 0
    ))
df_all['flag_lower_division'] = np.where(
    (df_all['division'] == 'Juryo')     & (df_all['opponent_division'] == 'Maegashira'), 1, np.where(
    (df_all['division'] == 'Makushita') & (df_all['opponent_division'] == 'Juryo')     , 1, 0
    ))
df_all['flag_same_hometown'] = np.where(df_all['shusshin'] == df_all['opponent_shusshin'], 1, 0)
df_all['flag_first_day'] = np.where(df_all['day'] == 1, 1, 0)
df_all['flag_last_day'] = np.where(df_all['day'] >= 15, 1, 0)

In [26]:
# Create historical performance statistics
# because wrestlers can have the same names, we identify using multiple variables

# Calculate cumulative win rate for each wrestler up to each bout
df_all.sort_values(by=['name','shusshin','birth_date','basho','day','bout_id'], inplace=True)
df_all['cum_wins'] = df_all.groupby(['name','shusshin','birth_date'])['win'].cumsum() - df_all['win']
df_all['cum_bouts'] = df_all.groupby(['name','shusshin','birth_date']).cumcount()
df_all['cum_win_rate'] = np.where(df_all['cum_bouts'] > 0, df_all['cum_wins'] / df_all['cum_bouts'], np.nan)

# Calculate win rate for previous 6 tournaments for each wrestler
df_all['basho_dt'] = pd.to_datetime(df_all['basho'].astype(str), format='%Y.%m')
df_all.sort_values(by=['name','shusshin','birth_date','basho_dt','day','bout_id'], inplace=True)

# Compute win rate per tournament
tournament_stats = df_all.groupby(['name','shusshin','birth_date','basho','basho_dt']).agg(
    wins=('win','sum'),
    bouts=('win','count')
).reset_index()

# Rolling aggregate for previous 6 tournaments (excluding current)
tournament_stats['wins_prev6'] = tournament_stats.groupby(['name','shusshin','birth_date'])['wins'].transform(
    lambda x: x.shift(1).rolling(window=6, min_periods=1).sum()
)
tournament_stats['bouts_prev6'] = tournament_stats.groupby(['name','shusshin','birth_date'])['bouts'].transform(
    lambda x: x.shift(1).rolling(window=6, min_periods=1).sum()
)
tournament_stats['win_rate_prev6'] = np.where(
    tournament_stats['bouts_prev6'] > 0,
    tournament_stats['wins_prev6'] / tournament_stats['bouts_prev6'],
    np.nan
)

# Merge back to df_all
df_all = pd.merge(df_all, tournament_stats[['name','shusshin','birth_date', 'basho', 'wins_prev6', 'bouts_prev6', 'win_rate_prev6']],
                  on=['name','shusshin','birth_date','basho'], how='left')

df_all.drop(columns=['basho_dt'], inplace=True)

In [27]:
# Now we need to join the win rate fields for opponents as well
winrate_opp = df_all[['name', 'bout_id', 'cum_wins', 'cum_bouts',
       'cum_win_rate', 'wins_prev6', 'bouts_prev6',
       'win_rate_prev6']]

winrate_opp.columns = [col if col in ['bout_id'] else 'opponent_'+col for col in winrate_opp.columns]

df_all = pd.merge(df_all, winrate_opp, on=['opponent_name','bout_id'], how='left', validate='1:1')

In [28]:
df_all = df_all.dropna()
df_all.to_pickle('../data/cleaned_sumo_bouts.pkl')
df_all

,basho,day,bout_id,technique,win,name,heya,shusshin,birth_date,age,...,cum_win_rate,wins_prev6,bouts_prev6,win_rate_prev6,opponent_cum_wins,opponent_cum_bouts,opponent_cum_win_rate,opponent_wins_prev6,opponent_bouts_prev6,opponent_win_rate_prev6
16,2015.05,2,45227,hatakikomi,1,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,216,438,0.493151,40.0,77.0,0.519481
17,2015.05,3,45264,sukuinage,0,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.529412,7.0,15.0,0.466667,83,165,0.503030,40.0,83.0,0.481928
18,2015.05,4,45295,tsukiotoshi,0,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,3,6,0.500000,2.0,3.0,0.666667
19,2015.05,5,45329,uwatenage,1,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.473684,7.0,15.0,0.466667,11,26,0.423077,8.0,22.0,0.363636
20,2015.05,6,45365,hatakikomi,1,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,117,266,0.439850,33.0,76.0,0.434211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152131,2022.11,11,67538,yorikiri,0,Yutakayama,Tokitsukaze,Niigata,1993-09-22,29.1,...,0.465696,38.0,90.0,0.422222,208,409,0.508557,39.0,76.0,0.513158
152132,2022.11,12,67572,yorikiri,0,Yutakayama,Tokitsukaze,Niigata,1993-09-22,29.1,...,0.464730,38.0,90.0,0.422222,302,599,0.504174,42.0,88.0,0.477273
152133,2022.11,13,67605,yorikiri,1,Yutakayama,Tokitsukaze,Niigata,1993-09-22,29.1,...,0.463768,38.0,90.0,0.422222,472,977,0.483112,43.0,90.0,0.477778
152134,2022.11,14,67640,yorikiri,0,Yutakayama,Tokitsukaze,Niigata,1993-09-22,29.1,...,0.464876,38.0,90.0,0.422222,33,71,0.464789,27.0,58.0,0.465517


In [29]:
df_all.columns

Index(['basho', 'day', 'bout_id', 'technique', 'win', 'name', 'heya',
       'shusshin', 'birth_date', 'age', 'division', 'height', 'weight',
       'experience', 'rank', 'rank_num', 'opponent_name', 'opponent_heya',
       'opponent_shusshin', 'opponent_age', 'opponent_division',
       'opponent_height', 'opponent_weight', 'opponent_experience',
       'opponent_rank', 'opponent_rank_num', 'age_diff', 'height_diff',
       'weight_diff', 'experience_diff', 'rank_diff', 'flag_higher_division',
       'flag_lower_division', 'flag_same_hometown', 'flag_first_day',
       'flag_last_day', 'cum_wins', 'cum_bouts', 'cum_win_rate', 'wins_prev6',
       'bouts_prev6', 'win_rate_prev6', 'opponent_cum_wins',
       'opponent_cum_bouts', 'opponent_cum_win_rate', 'opponent_wins_prev6',
       'opponent_bouts_prev6', 'opponent_win_rate_prev6'],
      dtype='object')

In [30]:
# Create an aggregated df by tournament/basho
df_tournament = df_all.groupby(['basho','name','heya','shusshin','age', 'division', 
    'intai', 'height', 'weight', 'experience', 'rank_num']).agg({
        'wins': ('win','sum'),
        'bouts': ('win','count'),
        ['opponent_rank_num', 'age_diff', 'height_diff', 'weight_diff',
       'experience_diff', 'rank_diff', 'flag_higher_division',
       'flag_lower_division', 'flag_same_hometown', 'cum_wins', 'cum_bouts',
       'cum_win_rate', 'wins_prev6', 'bouts_prev6', 'win_rate_prev6',
       'opponent_cum_wins', 'opponent_cum_bouts', 'opponent_cum_win_rate',
       'opponent_wins_prev6', 'opponent_bouts_prev6',
       'opponent_win_rate_prev6']: 'mean'
})
df_tournament

KeyError: 'intai'